In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option("display.max_colwidth", 90)

# Task 1

In [ ]:
CSV_PATH = "spam.csv"
MIRROR_URL = ("https://raw.githubusercontent.com/mohitgupta-omg/"
              "Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv")

if not os.path.exists(CSV_PATH):
    print("spam.csv not found locally -> downloading mirror copy of the Kaggle file...")
    urllib.request.urlretrieve(MIRROR_URL, CSV_PATH)

df_raw = pd.read_csv(CSV_PATH, encoding="latin-1")
print("Raw shape:", df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()
print("\nColumns:", df_raw.columns.tolist())

In [ ]:
print("Non-null counts in artifact columns:")
print(df_raw[["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]].notna().sum())

In [ ]:
# Keep only the primary columns v1 (label) and v2 (text) by dropping the empty remnants
df = df_raw.drop(columns=["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"])

# Rename to meaningful names
df = df.rename(columns={"v1": "label", "v2": "message_text"})

# Encode target: ham -> 0 (legitimate), spam -> 1 (unsolicited)
df["label"] = df["label"].map({"ham": 0, "spam": 1})

print("Clean shape:", df.shape)
print("Columns    :", df.columns.tolist())
print("Label dtype:", df["label"].dtype)
df.head()

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print("\nTotal missing:", int(missing.sum()))
print("Duplicate messages:", int(df.duplicated().sum()), "(kept as-is; the task does not ask for de-duplication)")

In [ ]:
counts = df["label"].value_counts().sort_index()
percent = df["label"].value_counts(normalize=True).sort_index() * 100

dist = pd.DataFrame({
    "class": ["ham (0)", "spam (1)"],
    "count": counts.values,
    "percentage": percent.round(2).values,
})
print(dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(["ham (0)", "spam (1)"], counts.values, color=["#2b7a4b", "#c0392b"])
for b, p in zip(bars, percent.values):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 40, f"{p:.1f}%",
            ha="center", fontweight="bold")
ax.set_title("Class Distribution")
ax.set_ylabel("Number of messages")
ax.set_ylim(0, counts.max() * 1.12)
plt.tight_layout()
plt.show()

In [ ]:
df["char_length"] = df["message_text"].str.len()

avg_len = df.groupby("label")["char_length"].mean()
print(f"Average length of HAM  messages : {avg_len[0]:.2f} characters")
print(f"Average length of SPAM messages : {avg_len[1]:.2f} characters")
print(f"Spam messages are on average {avg_len[1] - avg_len[0]:.1f} characters longer "
      f"({avg_len[1] / avg_len[0]:.2f}x).")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(df.loc[df.label == 0, "char_length"], bins=50, alpha=0.7, label="ham", color="#2b7a4b")
ax.hist(df.loc[df.label == 1, "char_length"], bins=50, alpha=0.7, label="spam", color="#c0392b")
ax.set_xlabel("Message length (characters)")
ax.set_ylabel("Count")
ax.set_title("Message Length Distribution")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
X = df["message_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=df["label"],
)

print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")
split_check = pd.DataFrame({
    "train_spam_%": [y_train.mean() * 100],
    "test_spam_%":  [y_test.mean() * 100],
    "full_spam_%":  [y.mean() * 100],
}).round(2)
print("\nSpam ratio check (should be nearly identical):")
print(split_check.to_string(index=False))

#Task2

In [ ]:
tfidf = TfidfVectorizer(lowercase=True, stop_words="english")

# Fit ONLY on training text, then transform train
X_train_tfidf = tfidf.fit_transform(X_train)

# Test set: transform only (no fitting) -> no leakage
X_test_tfidf = tfidf.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf  shape:", X_test_tfidf.shape)
print("Vocabulary size    :", len(tfidf.vocabulary_))
sparsity = 1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print(f"Sparsity (train)   : {sparsity:.4%}")

In [ ]:
nb_model = MultinomialNB()          # default alpha = 1.0
nb_model.fit(X_train_tfidf, y_train)

print("Model trained.")
print("Laplace smoothing alpha:", nb_model.alpha)
print("Classes:", nb_model.classes_)

In [ ]:
y_pred = nb_model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}  ({acc * 100:.2f}%)")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix (rows = actual, cols = predicted):")
print(pd.DataFrame(cm, index=["Actual HAM", "Actual SPAM"],
                       columns=["Pred HAM", "Pred SPAM"]))
print(f"\nTrue Negatives  (ham  -> ham ) : {tn}")
print(f"False Positives (ham  -> SPAM) : {fp}   <- legitimate messages wrongly blocked")
print(f"False Negatives (spam -> ham ) : {fn}   <- spam that slipped through")
print(f"True Positives  (spam -> SPAM) : {tp}")

fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["HAM", "SPAM"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["HAM", "SPAM"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Test Set")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16, fontweight="bold",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=["HAM (0)", "SPAM (1)"], digits=4))

In [ ]:
def predict_message(text: str) -> str:
    """Classify one SMS using the fitted TF-IDF vectorizer + Naive Bayes model."""
    features = tfidf.transform([text])
    pred = nb_model.predict(features)[0]
    return "SPAM" if pred == 1 else "HAM"

def spam_probability(text: str) -> float:
    return float(nb_model.predict_proba(tfidf.transform([text]))[0, 1])

samples = [
    "Hey, are we still meeting for lunch today?",
    "Congratulations! You have won a $1,000 gift card. Call now to claim.",
]

print(f"{'Predicted':<10} | {'P(spam)':<8} | Message")
print("-" * 90)
for msg in samples:
    print(f"{predict_message(msg):<10} | {spam_probability(msg):<8.3f} | {msg}")

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
log_ratio = nb_model.feature_log_prob_[1] - nb_model.feature_log_prob_[0]

top_spam = feature_names[np.argsort(log_ratio)[::-1][:15]]
top_ham = feature_names[np.argsort(log_ratio)[:15]]

print("Top 15 SPAM-indicative words:", ", ".join(top_spam))
print("Top 15 HAM-indicative words :", ", ".join(top_ham))